# Comparing Efficient Multi-Head Attention Implementations

This code notebook compares different ways to implement causal multi-head attention used in decoder-style LLMs like GPT, Llama, etc.

In [1]:
import torch

torch.manual_seed(123)
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device =  torch.device("cpu")

print(f"Using device: {device}")
print(f"Pytorch Version: {torch.__version__}")

batch_size = 8
context_len  = 1024
embed_dim = 768
embeddings = torch.randn((batch_size, context_len, embed_dim), device=device)

Using device: cuda
Pytorch Version: 2.5.1+cu121


- To run all the code in this notebook, please ensure you update to at least PyTorch 2.5 (FlexAttention is not included in earlier PyTorch releases)
- If the code cell above shows a PyTorch version lower than 2.5, you can upgrade your PyTorch installation by uncommenting and running the following code cell (Please note that PyTorch 2.5 requires Python 3.9 or later)
- For more specific instructions and CUDA versions, please refer to the official installation guide at https://pytorch.org

<br>
&nbsp;

## 1. CausalAttention MHA wrapper class from chapter 3

     Menial implementation of Multi-Head Attention

In [16]:
import torch.nn as nn


class CasualAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # register buffer is used to move the tensors to the cuda device; not the default feature
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights  = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values 
        return context_vec
    

class MHA_Wrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CasualAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

        self.out_proj = nn.Linear(d_out * num_heads, d_out * num_heads)

    def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)
    
mha = MHA_Wrapper(
    d_in=embed_dim,
    d_out=embed_dim//12,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha(embeddings)
print(out.shape)
print(out)

torch.Size([8, 1024, 768])
tensor([[[-1.9849e-01,  1.0589e-01, -2.8424e-01,  ...,  3.7548e-01,
          -9.0961e-01,  2.2660e-01],
         [ 2.0488e-01,  4.9862e-02, -8.2041e-02,  ...,  2.2870e-02,
          -5.8894e-01,  1.8700e-01],
         [ 2.8205e-01,  3.3025e-02,  1.0439e-01,  ...,  1.0868e-01,
          -4.7152e-01,  9.7241e-02],
         ...,
         [-4.0142e-02, -4.8613e-02,  4.9493e-02,  ...,  5.1844e-03,
           2.0382e-02, -1.0297e-02],
         [-4.0000e-02, -4.6687e-02,  4.6737e-02,  ...,  1.4811e-03,
           1.3583e-02, -2.3320e-03],
         [-3.7209e-02, -4.2204e-02,  5.2374e-02,  ..., -8.3704e-03,
           1.8733e-02,  1.0116e-03]],

        [[ 5.3827e-01, -4.6380e-02, -2.6095e-01,  ..., -4.7337e-02,
          -1.5414e-01, -6.3111e-01],
         [ 6.4156e-02,  2.5731e-01, -1.2166e-01,  ...,  2.7246e-02,
           1.5609e-01, -4.1344e-01],
         [ 7.4470e-02,  7.2050e-02,  5.8388e-03,  ...,  1.5305e-01,
          -2.5993e-02, -2.7842e-01],
         ...